# Vendor Revenue Forecast
Multiple Linear Regression — Predict per-vendor daily revenue.

Run cells top-to-bottom: **Setup → Generate Data → Train → Evaluate → Test**

## 1. Setup
Resolve paths for `data/`, `output/`, `report/` directories.

In [ ]:
from pathlib import Path

try:
    NOTEBOOK_DIR = Path(__vsc_ipynb_file__).resolve().parent
except NameError:
    NOTEBOOK_DIR = Path().resolve()

BASE_DIR   = NOTEBOOK_DIR.parent
DATA_DIR   = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "output"
REPORT_DIR = BASE_DIR / "report"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

print(f"BASE_DIR   : {BASE_DIR}")
print(f"DATA_DIR   : {DATA_DIR}")
print(f"OUTPUT_DIR : {OUTPUT_DIR}")
print(f"REPORT_DIR : {REPORT_DIR}")

## 2. Generate Data
Create synthetic training data and save to `data/`.

In [ ]:
import csv
import random
import math
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = SCRIPT_DIR.parent / "data"
DATA_DIR.mkdir(parents=True, exist_ok=True)

random.seed(42)

def normal_noise(mu=0, sigma=1):
    u1 = random.random()
    u2 = random.random()
    z = math.sqrt(-2 * math.log(u1)) * math.cos(2 * math.pi * u2)
    return mu + sigma * z

vendors = [
    {"name": "Everyday Essentials", "base": 250, "wknd_lift": 180},
    {"name": "Casa Living",          "base": 230, "wknd_lift": 170},
    {"name": "Home & More",          "base": 210, "wknd_lift": 160},
    {"name": "The Linen Store",      "base": 190, "wknd_lift": 140},
    {"name": "Street Fusion",        "base": 120, "wknd_lift": 100},
]

dow_effect = [-2, -1, 0, 1, 2, 3, 4]  # Mon=0 ... Sun=6

rows = []
for vendor in vendors:
    for day in range(90):
        dow = day % 7
        is_weekend = 1 if dow >= 5 else 0
        week_num = day // 7
        revenue = (vendor["base"]
                   + vendor["wknd_lift"] * is_weekend
                   + 5 * dow_effect[dow]
                   + normal_noise(0, 30))
        rows.append({
            "vendor_name": vendor["name"],
            "day_of_week": dow,
            "is_weekend": is_weekend,
            "week_num": week_num,
            "revenue": round(revenue, 2)
        })

out_path = DATA_DIR / "vendor_revenue_data.csv"
with open(out_path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f, fieldnames=["vendor_name", "day_of_week", "is_weekend", "week_num", "revenue"]
    )
    writer.writeheader()
    writer.writerows(rows)

print(f"Saved {len(rows)} rows to {out_path}")

## 3. Train
Fit the model and save to `output/`.

In [ ]:
import csv
import pickle
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
DATA_DIR = SCRIPT_DIR.parent / "data"
OUTPUT_DIR = SCRIPT_DIR.parent / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def _T(A):
    return [[A[j][i] for j in range(len(A))] for i in range(len(A[0]))]

def _mul(A, B):
    m, n, p = len(A), len(A[0]), len(B[0])
    return [[sum(A[i][k] * B[k][j] for k in range(n)) for j in range(p)] for i in range(m)]

def _inv(A):
    n = len(A)
    M = [A[i][:] + [1.0 if i == j else 0.0 for j in range(n)] for i in range(n)]
    for col in range(n):
        pivot = max(range(col, n), key=lambda r: abs(M[r][col]))
        M[col], M[pivot] = M[pivot], M[col]
        M[col] = [x / M[col][col] for x in M[col]]
        for row in range(n):
            if row != col:
                f = M[row][col]
                M[row] = [M[row][k] - f * M[col][k] for k in range(2 * n)]
    return [M[i][n:] for i in range(n)]

def fit_lr(X, y):
    Xb = [[1] + row for row in X]
    Xt = _T(Xb)
    theta = _mul(_inv(_mul(Xt, Xb)), _mul(Xt, [[v] for v in y]))
    return [t[0] for t in theta]

def predict_lr(theta, X):
    return [sum(t * x for t, x in zip(theta, [1] + row)) for row in X]

# ---------- Load data ----------
rows = []
with open(DATA_DIR / "vendor_revenue_data.csv", newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        rows.append(row)

# Group by vendor
vendors = {}
for row in rows:
    name = row["vendor_name"]
    if name not in vendors:
        vendors[name] = {"X": [], "y": []}
    vendors[name]["X"].append([
        int(row["day_of_week"]),
        int(row["is_weekend"]),
        int(row["week_num"])
    ])
    vendors[name]["y"].append(float(row["revenue"]))

models = {}
test_data = {}
for name, data in vendors.items():
    X, y = data["X"], data["y"]
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]
    theta = fit_lr(X_train, y_train)
    models[name] = theta
    test_data[name] = {"X_test": X_test, "y_test": y_test}
    print(f"{name}: theta={[round(t, 2) for t in theta]}")

out_model = {
    "models": models,
    "feature_cols": ["day_of_week", "is_weekend", "week_num"],
    "test_data": test_data
}

out_path = OUTPUT_DIR / "vendor_revenue_models.pkl"
with open(out_path, "wb") as f:
    pickle.dump(out_model, f)

print(f"\nModel saved to {out_path}")

## 4. Evaluate
Compute metrics on the test set and save to `report/`.

In [ ]:
import pickle
import json
import math
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"
REPORT_DIR = SCRIPT_DIR.parent / "report"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

def predict_lr(theta, X):
    return [sum(t * x for t, x in zip(theta, [1] + row)) for row in X]

def r2(y_true, y_pred):
    mean_y = sum(y_true) / len(y_true)
    ss_res = sum((t - p) ** 2 for t, p in zip(y_true, y_pred))
    ss_tot = sum((t - mean_y) ** 2 for t in y_true)
    return 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

def mae(y_true, y_pred):
    return sum(abs(t - p) for t, p in zip(y_true, y_pred)) / len(y_true)

with open(OUTPUT_DIR / "vendor_revenue_models.pkl", "rb") as f:
    out_model = pickle.load(f)

models = out_model["models"]
test_data = out_model["test_data"]

vendors_report = {}
all_y_true = []
all_y_pred = []

for name, theta in models.items():
    X_test = test_data[name]["X_test"]
    y_test = test_data[name]["y_test"]
    y_pred = predict_lr(theta, X_test)
    r2_val = round(r2(y_test, y_pred), 4)
    mae_val = round(mae(y_test, y_pred), 4)
    vendors_report[name] = {"r2": r2_val, "mae": mae_val}
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    print(f"{name}: R²={r2_val:.4f}  MAE={mae_val:.4f}")

overall_r2 = round(r2(all_y_true, all_y_pred), 4)
print(f"\nOverall R²={overall_r2:.4f}")

report = {
    "model": "Multiple Linear Regression",
    "task": "Vendor daily revenue forecast",
    "vendors": vendors_report,
    "overall_r2": overall_r2
}

out_path = REPORT_DIR / "evaluation_report.json"
with open(out_path, "w") as f:
    json.dump(report, f, indent=2)

print(f"Report saved to {out_path}")

## 5. Test
Run inference on sample inputs.

In [ ]:
import pickle
from pathlib import Path

SCRIPT_DIR = Path(__file__).resolve().parent
OUTPUT_DIR = SCRIPT_DIR.parent / "output"

def predict_lr(theta, X):
    return [sum(t * x for t, x in zip(theta, [1] + row)) for row in X]

with open(OUTPUT_DIR / "vendor_revenue_models.pkl", "rb") as f:
    out_model = pickle.load(f)

models = out_model["models"]

days = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
week_num = 6  # mid-period week

print("\n=== Vendor Revenue Forecast: Mon-Sun Predictions (week 6) ===")
header = f"{'Day':<5}" + "".join(f"{name[:12]:>14}" for name in models)
print(header)

for dow in range(7):
    is_weekend = 1 if dow >= 5 else 0
    row_str = f"{days[dow]:<5}"
    for name, theta in models.items():
        pred = predict_lr(theta, [[dow, is_weekend, week_num]])[0]
        row_str += f"  ${pred:>10,.2f}"
    print(row_str)